# **Logistic Regression: Stroke Predictions** #

## Objectives

* Implement a machine learning algorithm using SciKit-learn to predict whether a patient has had a stroke or not

## Inputs

* Cleaned_Dataset.csv 

## Outputs

* ML Performance during training and testing



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\Lailah\\vscode-project\\DA2_Assessment\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\Lailah\\vscode-project\\DA2_Assessment'

# ML Pipeline

Section 1 content

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [125]:
df = pd.read_csv("../Data/Cleaned_Data/Cleaned_Data.csv")
df.head(8)

,hypertension,heart_disease,bmi,weight_category,avg_glucose_level,stroke,gender,gender_Male,gender_Other,age,ever_married,ever_married_Yes,Residence_type,Residence_type_Urban,work_type,work_type_encoded
0,0,1,36.6,Obese,169.3575,1,Male,1.0,0.0,67.0,Yes,1.0,Urban,1.0,Private,2.0
1,0,0,28.1,Overweight,169.3575,1,Female,0.0,0.0,61.0,Yes,1.0,Rural,0.0,Self-employed,3.0
2,0,1,32.5,Obese,105.9200,1,Male,1.0,0.0,80.0,Yes,1.0,Rural,0.0,Private,2.0
3,0,0,34.4,Obese,169.3575,1,Female,0.0,0.0,49.0,Yes,1.0,Urban,1.0,Private,2.0
4,1,0,24.0,Healthy,169.3575,1,Female,0.0,0.0,79.0,Yes,1.0,Rural,0.0,Self-employed,3.0
5,0,0,29.0,Overweight,169.3575,1,Male,1.0,0.0,81.0,Yes,1.0,Urban,1.0,Private,2.0
6,1,1,27.4,Overweight,70.0900,1,Male,1.0,0.0,74.0,Yes,1.0,Rural,0.0,Private,2.0
7,0,0,22.8,Healthy,94.3900,1,Female,0.0,0.0,69.0,No,0.0,Urban,1.0,Private,2.0


In [126]:
df.columns

Index(['hypertension', 'heart_disease', 'bmi', 'weight_category',
       'avg_glucose_level', 'stroke', 'gender', 'gender_Male', 'gender_Other',
       'age', 'ever_married', 'ever_married_Yes', 'Residence_type',
       'Residence_type_Urban', 'work_type', 'work_type_encoded'],
      dtype='object')

Removed all columns that have string data as this is not compatible with Logistic Regression model. This isnt an issues as all string variables have been encoded so their value is still present

In [127]:
df = df.drop(
    columns=[
        "gender",
        "ever_married",
        "work_type",
        "Residence_type",
        'weight_category',
    ],
    #errors="ignore",
)
df.head()

,hypertension,heart_disease,bmi,avg_glucose_level,stroke,gender_Male,gender_Other,age,ever_married_Yes,Residence_type_Urban,work_type_encoded
0,0,1,36.6,169.3575,1,1.0,0.0,67.0,1.0,1.0,2.0
1,0,0,28.1,169.3575,1,0.0,0.0,61.0,1.0,0.0,3.0
2,0,1,32.5,105.9200,1,1.0,0.0,80.0,1.0,0.0,2.0
3,0,0,34.4,169.3575,1,0.0,0.0,49.0,1.0,1.0,2.0
4,1,0,24.0,169.3575,1,0.0,0.0,79.0,1.0,0.0,3.0


**Test and Train**
* Split dataset into train and test sets. Random state was selcted as 101 for preproducability purposes

In [128]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.drop(["stroke"],axis=1), df["stroke"], test_size=0.15, random_state=101,stratify=df["stroke"]
)

print(
    "* Train set:",
    X_train.shape,
    y_train.shape,
    "\n* Test set:",
    X_test.shape,
    y_test.shape,
)

* Train set: (4343, 10) (4343,) 
* Test set: (767, 10) (767,)


**Create a pipeline - Binary Classifier (Logistic Regression)**

In [130]:
from sklearn.pipeline import Pipeline

##Feature Scaling 
from sklearn.preprocessing import StandardScaler

##Feature Selection
from sklearn.feature_selection import SelectFromModel

##ML Algorithms
from sklearn.linear_model import LogisticRegression


In [131]:
def pipeline_logistic_regression():
    pipeline = Pipeline(
        [
            ("scaler", StandardScaler()),
            
            # Feature selection using a balanced logistic regression
            ("feature_selection", 
             SelectFromModel(
                 LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
             )
            ),

            # Final model also using balanced logistic regression
            ("model", 
             LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000)
            ),
        ]
    )
    return pipeline


**Fitting the pipeline to the train set**

In [132]:
pipeline=pipeline_logistic_regression()
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('feature_selection',
                 SelectFromModel(estimator=LogisticRegression(class_weight='balanced',
                                                              max_iter=1000,
                                                              random_state=42))),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

**Evaluate Pipeline**

In [133]:
def logistic_regression_coef(model,columns):
    coeff_df=pd.DataFrame(
        model.coef_, index=["Coefficient"], columns=columns
    ).T.sort_values(["Coefficient"], key=abs, ascending=False)
    print(coeff_df)

In [134]:
logistic_regression_coef(
    model=pipeline["model"],
    columns=X_train.columns[pipeline["feature_selection"].get_support()],
)

     Coefficient
age     1.725437


In [135]:
from sklearn.metrics import classification_report, confusion_matrix
def confusion_matrix_and_report(X,y, pipeline, label_map):
    prediction=pipeline.predict(X)

    print("--- Confusion Matrix ---")
    print(
        pd.DataFrame(
            confusion_matrix(y_true=prediction, y_pred=y),
            columns=[["Actual" + sub for sub in label_map]], 
            index=[["Prediction" + sub for sub in label_map]],
        )
    )
    print("\n")

    print("--- Classification Report ---")
    print(classification_report(y, prediction, target_names=label_map), "\n")

In [136]:
def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map):
    print("###Train set performance###\n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("###Test set performance###\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

In [137]:
clf_performance(
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    pipeline=pipeline,
    label_map=["No Stroke", "Stroke"],
)

###Train set performance###

--- Confusion Matrix ---
                    ActualNo Stroke ActualStroke
PredictionNo Stroke            2936           38
PredictionStroke               1195          174


--- Classification Report ---
              precision    recall  f1-score   support

   No Stroke       0.99      0.71      0.83      4131
      Stroke       0.13      0.82      0.22       212

    accuracy                           0.72      4343
   macro avg       0.56      0.77      0.52      4343
weighted avg       0.95      0.72      0.80      4343
 

###Test set performance###

--- Confusion Matrix ---
                    ActualNo Stroke ActualStroke
PredictionNo Stroke             513            4
PredictionStroke                217           33


--- Classification Report ---
              precision    recall  f1-score   support

   No Stroke       0.99      0.70      0.82       730
      Stroke       0.13      0.89      0.23        37

    accuracy                           0.7

---

**Results/Summary**

Logistic Regression was chosen as the learning algorithm because the target variable (Stroke) is categorical, making this a binary classification problem. Logistic Regression is specifically designed for binary outcomes.  
Other algorithms such as Decision Trees or Random Forests could also have been used and may offer better performance, but exploring these models is outside the current scope of my knowledge.

When deciding how to split the test and train set different combination where chosen such as
Test: Train
0.15: 0.85
0.20: 0.80
0.30: 0.70
SMOTE 

Ultimately, I selected the 0.15 : 0.85 split. While its precision and recall were similar to the 0.20 split, the 0.15 split achieved slightly better accuracy and produced fewer false positives and false negatives.  
Both the 0.30 split and the SMOTE‑only evaluation performed worse and were less reliable.

RESULTS
Train Set Performance 
The precision and recall for no Stoke was high at 99% and 71% respectievly meaning the algorithm was able to accuratly predict 99% of the as No Stroke. and the model correctly identified 71% of all actual No Stroke cases.

From the confusion matrix
correct No‑Stroke = 2936
predicted Stroke but actually No‑Stroke = 1195
missed Stroke = 38
correct Stroke = 174
The model catches 82% of real stroke cases (recall = 0.82)meaning the model correctly identified 82% of real stroke cases. But only 13% of predicted strokes are actually strokes (precision = 0.13). This shows that the model is very senstive to detecting stroke cases but is poor at predicting stroke cases with most prediction giving false positives. 

The poorer performance for Stoke over non-Stroke cases is most likely down to the uneven distibution of data. In a dataset of 5110 only 249 cases are stroke, so in order for this models performance to be improved either more stroke cases need to be added to the dataset or another algorithm be implemented 



---

---